# IRF systematics study: fixed counts, perturbed IRFs

The previous notebook on the joint crab shows that you can add nuissance parameters in the fit with custom models to take into account the systematic uncertainties.

This notebook shows that for modifications that are not trivial to include as models, we can create samples of datasets with modified IRFs and marginalise over realisation to get the systematic errors.


The idea is to keep the counts fixed and perturb the IRFs used in the fit:

- the counts are fixed
- draw random IRF variations (background norm, tilt, spatial gradient,
     and an energy-scale shift of exposure/edisp/psf).
- the same counts are re-fitted with the perturbed IRFs
- the spread of the fitted source parameters around the nominal fit is the systematic uncertainty.

In [1]:
import sys
import numpy as np
import pandas as pd
from regions import CircleSkyRegion, RectangleSkyRegion
from astropy.coordinates import SkyCoord, Angle
import astropy.units as u

from gammapy.catalog import SourceCatalogHGPS
from gammapy.datasets import MapDataset
from gammapy.maps import Map, MapAxis, WcsGeom
from gammapy.modeling import Fit
from gammapy.modeling.models import (
    Models,
    PowerLawNormSpectralModel,
    FoVBackgroundModel,
)

## Configuration

Here we setup the perturbation parameters for the IRFs. 

In [2]:
# Widths of the random IRF variations (Gaussian draws)
var_norm_bkg = 0.01         # relative background normalisation
var_tilt_bkg = 0.02         # background spectral tilt
var_grad_bkg_amp = 0.01     # background gradient amplitude [per deg]
var_e_shift = 0.1           # energy-scale shift (log-normal width)
nsets = 5                   # perturbed IRF sets per seed

var_names = ["norm_bkg", "tilt_bkg", "grad_bkg_amp", "grad_bkg_angle", "e_shift"]
savedir = "varied_irf_results"

print("Initial parameters:")
print("var_norm_bkg     =", var_norm_bkg)
print("var_tilt_bkg     =", var_tilt_bkg)
print("var_grad_bkg_amp =", var_grad_bkg_amp)
print("var_e_shift      =", var_e_shift)
print("nsets            =", nsets)

Initial parameters:
var_norm_bkg     = 0.01
var_tilt_bkg     = 0.02
var_grad_bkg_amp = 0.01
var_e_shift      = 0.1
nsets            = 5


## Load catalog, dataset, models

In [3]:
# Stacked dataset (counts, background, exposure, psf, edisp, mask_safe)
dataset = MapDataset.read("datasets/rxj/hess_rxj_stacked.fits.gz", name="stacked")

geom = dataset.counts.geom.copy()
geom_image = geom.to_image()  # spatial-only geometry, used for the gradient

models = Models.read("models_rxj.yaml")
source_model = models[0]

# Sanity bounds so the fit cannot wander into unphysical values
source_model.parameters["amplitude"].min = 0
source_model.parameters["index"].min = 1
source_model.parameters["index"].max = 5

print("Input model parameters:")
print(source_model.parameters.to_table())

# Keep the true (input) source parameters so pulls can be computed later
models.parameters.to_table().to_pandas().to_csv(
    f"{savedir}/input_parameters.csv", index=False
)

Input model parameters:
type    name      value         unit      ...    max    frozen link prior
---- --------- ----------- -------------- ... --------- ------ ---- -----
         index  2.2739e+00                ... 5.000e+00  False           
     amplitude  2.3752e-11 TeV-1 s-1 cm-2 ...       nan  False           
     reference  1.0000e+00            TeV ...       nan   True           
         lon_0  3.4725e+02            deg ... 3.478e+02  False           
         lat_0 -3.8801e-01            deg ... 2.431e-01  False           
         sigma  3.4578e-01            deg ...       nan  False           
             e  0.0000e+00                ... 1.000e+00   True           
           phi  0.0000e+00            deg ... 1.800e+02   True           


In [4]:
source_position = SkyCoord.from_name("RX J1713.7-3946").galactic

# Exclusion for known HESS sources
cat = SourceCatalogHGPS('$GAMMAPY_DATA/catalogs/hgps_catalog_v1.fits.gz')
regions = []
for src in cat:
    if src.position.separation(source_position) < 5*u.deg:
        regions.append(CircleSkyRegion(center=src.position, radius=src.data.RSpec * 2.4))
# Add mask around Galactic Plane
region_galactic = RectangleSkyRegion(center=SkyCoord(source_position.galactic.l, 0*u.deg, frame='galactic'), 
                                     width=geom.width[0][0]*2.1, 
                                     height=1.4*u.deg)
regions.append(region_galactic)
exclusion_mask = geom.region_mask(regions, inside=False)
dataset.mask_safe.data &= exclusion_mask.data

### Geometry for the background gradient (computed once)


In [5]:
coords = geom_image.get_coord().skycoord      # sky position of each pixel
center = geom_image.center_skydir             # map centre
pos_angle = center.position_angle(coords)     # position angle of each pixel
offset = center.separation(coords)            # angular distance from centre

## Function for energy-shifted copies of the IRFs

In [6]:
def apply_energy_shift(ds, e_shift):
    """Shift the true-energy axis of exposure, edisp and psf by e_shift."""
    # New true-energy axis with all edges scaled by e_shift
    new_e_axis = MapAxis.from_edges(
        ds.exposure.geom.axes[0].edges * e_shift,
        unit="TeV", interp="log", name="energy_true",
    )

    # Exposure: same data, relabelled energy axis
    new_exp_geom = WcsGeom(
        ds.exposure.geom.wcs, ds.exposure.geom.npix, axes=[new_e_axis]
    )
    ds.exposure = Map.from_geom(
        new_exp_geom, data=ds.exposure.data, unit=ds.exposure.unit
    )

    # Energy dispersion: shift reco axis too, then interpolate onto new geom
    old_edisp = ds.edisp.edisp_map
    new_energy_axis = MapAxis.from_edges(
        old_edisp.geom.axes[0].edges * e_shift, interp="log", name="energy"
    )
    new_edisp_geom = WcsGeom(
        old_edisp.geom.wcs, old_edisp.geom.npix,
        axes=[new_energy_axis, new_e_axis],
    )
    new_edisp_map = old_edisp.interp_to_geom(new_edisp_geom)
    new_edisp_map.data[new_edisp_map.data < 0] = 0   # interpolation can undershoot
    ds.edisp.edisp_map.data = new_edisp_map.data

    # PSF: same data, relabelled true-energy axis
    old_psf = ds.psf.psf_map
    new_psf_geom = WcsGeom(
        old_psf.geom.wcs, old_psf.geom.npix,
        axes=[old_psf.geom.axes[0].copy(), new_e_axis],
    )
    ds.psf.psf_map = Map.from_geom(
        new_psf_geom, data=old_psf.data, unit=old_psf.unit
    )

# Perform a nominal fit to get the start values

In [7]:
seed = 0
rsg = np.random.RandomState(seed)

In [8]:
# ------- Nominal fit: nominal IRFs, same (fixed) counts. Reference for the spread -------
nominal_ds = dataset.copy(name="nominal")

bkg_nom = FoVBackgroundModel(
    spectral_model=PowerLawNormSpectralModel(), dataset_name=nominal_ds.name
)
bkg_nom.spectral_model.norm.min = 0          # keep norm positive
bkg_nom.spectral_model.tilt.frozen = False   # fit the tilt as well
nominal_ds.models = Models([bkg_nom, source_model])

with nominal_ds.models.parameters.restore_status():
    fit = Fit(optimize_opts=dict(print_level=1))
    with np.errstate(divide="ignore", invalid="ignore"):
        result = fit.run([nominal_ds])
    if not result.success:
        raise RuntimeError("The nominal fit failed; check dataset/models first.")
    nominal_pars = result.parameters.to_table().to_pandas()

print("Nominal fit (nominal IRFs):")
print(nominal_pars)

nominal_pars.to_csv(f"{savedir}/nominal_fit.csv", index=False)

Nominal fit (nominal IRFs):
   type       name         value            unit         error         min  \
0             tilt  8.576046e-03                  1.983690e-02         NaN   
1             norm  9.836408e-01                  2.051630e-02    0.000000   
2        reference  1.000000e+00             TeV  0.000000e+00         NaN   
3            index  2.519417e+00                  4.945564e-01    1.000000   
4        amplitude  6.034628e-11  TeV-1 s-1 cm-2  2.247352e-11    0.000000   
5        reference  1.000000e+00             TeV  0.000000e+00         NaN   
6            lon_0  3.467694e+02             deg  1.324827e-03  346.769430   
7            lat_0 -2.930320e-01             deg  1.346121e-01   -0.756913   
8            sigma  7.627088e-01             deg  2.079686e-01    0.000000   
9                e  0.000000e+00                  0.000000e+00    0.000000   
10             phi  0.000000e+00             deg  0.000000e+00 -180.000000   

           max  frozen link prior  

# Run


Typically, this should be run a number of times to get a clear understanding of the systematics (but this takes a long time and requires computing power). Therefore, one should only do this once they are ready.

In [9]:
result_pars = []   # fitted parameter tables (successful fits only)
var_pars = []      # the random variations belonging to each successful fit
n_fail = 0

for i in range(nsets):
    print(f"\nGenerate perturbed IRF set {i + 1} / {nsets}")

    # ------- Random IRF variations -------
    norm_bkg = rsg.normal(1, var_norm_bkg)
    tilt_bkg = rsg.normal(0, var_tilt_bkg)
    grad_amp = rsg.normal(0, var_grad_bkg_amp)
    grad_angle = rsg.random_sample() * 2 * np.pi
    e_shift = np.exp(rsg.normal(0, var_e_shift))   # log-normal: always > 0

    print(f"  norm_bkg = {norm_bkg:.4g}  tilt_bkg = {tilt_bkg:.4g}")
    print(f"  grad_bkg_amp = {grad_amp:.4g}  grad_bkg_angle = {grad_angle:.4g}")
    print(f"  e_shift = {e_shift:.4g}")

    # ------- Build a dataset with the perturbed IRFs -------
    perturbed_dataset = dataset.copy(name="perturbed-dataset")

    # ------- Background gradient -------
    diff = pos_angle - Angle(grad_angle * u.rad)
    diff.wrap_at(2 * np.pi * u.rad, inplace=True)
    delta_lat = center.spherical_offsets_to(
        center.directional_offset_by(diff, offset)
    )[1]
    # Clip so large offsets cannot produce a negative background
    factors = np.clip(1 + delta_lat.to_value("deg") * grad_amp, 0, None)
    perturbed_dataset.background.data *= factors   # (y, x) broadcasts over the energy axis

    # ------- Energy-scale shift of exposure, edisp and psf -------
    apply_energy_shift(perturbed_dataset, e_shift)

    # ------- Background model carrying the varied norm and tilt -------
    bkg_true = FoVBackgroundModel(
        spectral_model=PowerLawNormSpectralModel(), dataset_name=perturbed_dataset.name
    )
    bkg_true.spectral_model.norm.value = norm_bkg
    bkg_true.spectral_model.tilt.value = tilt_bkg
    perturbed_dataset.models = Models([bkg_true, source_model])

    # Set the perturbed IRF dataset counts to the one of the original dataset 
    perturbed_dataset.counts = dataset.counts.copy()

    bkg_fit = FoVBackgroundModel(
        spectral_model=PowerLawNormSpectralModel(), dataset_name=dataset.name
    )
    bkg_fit.spectral_model.norm.min = 0          # keep norm positive
    bkg_fit.spectral_model.tilt.frozen = False   # fit the tilt as well
    dataset.models = Models([bkg_fit, source_model])

    # restore_status to reset the source parameters after each fit
    with models.parameters.restore_status():
        fit = Fit(optimize_opts=dict(print_level=1))
        with np.errstate(divide="ignore", invalid="ignore"):
            result = fit.run([dataset])

        if result.success:
            print("Fit successful!")
            result_pars.append(result.parameters.to_table().to_pandas())
            var_pars.append([norm_bkg, tilt_bkg, grad_amp, grad_angle, e_shift])
            print("Final model parameters:")
            print(dataset.models.parameters.to_table().to_pandas())
        else:
            n_fail += 1
            print("Fit FAILED - discarding this IRF set.")


Generate perturbed IRF set 1 / 5
  norm_bkg = 1.018  tilt_bkg = 0.008003
  grad_bkg_amp = 0.009787  grad_bkg_angle = 2.662
  e_shift = 1.251
W VariableMetricBuilder No improvement in line search
W MnPosDef Matrix forced pos-def by adding to diagonal 0.0732083
W VariableMetricBuilder Reached machine accuracy limit; Edm 0.000221138 is smaller than machine limit 0.00378064 while 0.0002 was requested
Fit successful!
Final model parameters:
   type       name         value            unit         error         min  \
0             tilt  1.076784e-02                  1.435910e-02         NaN   
1             norm  1.010951e+00                  1.231426e-02    0.000000   
2        reference  1.000000e+00             TeV  0.000000e+00         NaN   
3            index  2.580852e+00                  1.105046e+00    1.000000   
4        amplitude  5.182366e-10  TeV-1 s-1 cm-2  2.668197e-09    0.000000   
5        reference  1.000000e+00             TeV  0.000000e+00         NaN   
6            

/home/feijen/anaconda3/envs/gammapy-2.1/lib/python3.12/site-packages/astropy/units/quantity.py:648: RuntimeWarning: overflow encountered in power
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)
/home/feijen/anaconda3/envs/gammapy-2.1/lib/python3.12/site-packages/gammapy/maps/wcs/ndmap.py:1008: RuntimeWarning: Use of fft convolution on input with NAN or inf results in NAN or inf output. Consider using method='direct' instead.
  return scipy.signal.convolve(image, kernel, method=method, mode=mode)
/home/feijen/anaconda3/envs/gammapy-2.1/lib/python3.12/site-packages/gammapy/maps/wcs/ndmap.py:1008: RuntimeWarning: Use of fft convolution on input with NAN or inf results in NAN or inf output. Consider using method='direct' instead.
  return scipy.signal.convolve(image, kernel, method=method, mode=mode)
/home/feijen/anaconda3/envs/gammapy-2.1/lib/python3.12/site-packages/gammapy/maps/wcs/ndmap.py:1008: RuntimeWarning: Use of fft convolution on input with NAN or inf res

W VariableMetricBuilder No improvement in line search
W MnPosDef Matrix forced pos-def by adding to diagonal 0.0739677
Fit successful!
Final model parameters:
   type       name         value            unit         error         min  \
0             tilt  1.075500e-02                  1.440572e-02         NaN   
1             norm  1.010968e+00                  1.228624e-02    0.000000   
2        reference  1.000000e+00             TeV  0.000000e+00         NaN   
3            index  2.581981e+00                  1.149047e+00    1.000000   
4        amplitude  5.185305e-10  TeV-1 s-1 cm-2  2.660831e-09    0.000000   
5        reference  1.000000e+00             TeV  0.000000e+00         NaN   
6            lon_0  3.469850e+02             deg  1.684043e-01  346.769430   
7            lat_0  2.430795e-01             deg  3.061471e-03   -0.756913   
8            sigma  1.937323e-01             deg  2.337180e-01    0.000000   
9                e  0.000000e+00                  0.000000e+0

## Save results

In [10]:
import h5py

print(f"\n{n_fail} / {nsets} fits failed")

nominal_pars.to_csv(f"{savedir}/nominal_fit.csv", index=False)

if result_pars:
    values = np.array([t["value"].values for t in result_pars], dtype=float)   # (n_fits, n_pars)
    errors = np.array([t["error"].values for t in result_pars], dtype=float)   # (n_fits, n_pars)

    with h5py.File(f"{savedir}/results.h5", "w") as f:
        f.create_dataset("values", data=values)
        f.create_dataset("errors", data=errors)
        f.create_dataset("variations", data=np.array(var_pars, dtype=float))   # (n_fits, 5)
        f.create_dataset("nominal_values", data=nominal_pars["value"].values.astype(float))
        f.create_dataset("nominal_errors", data=nominal_pars["error"].values.astype(float))
        f.create_dataset("par_names", data=np.array(nominal_pars["name"], dtype="S"))
        f.create_dataset("var_names", data=np.array(var_names, dtype="S"))
    print(f"Saved {len(values)} fits to {savedir}/results.h5")
else:
    print("No successful fits - nothing saved.")


0 / 5 fits failed
Saved 5 fits to varied_irf_results/results.h5


## Results review

In [12]:
vals = np.array([t["value"].values for t in result_pars], dtype=float)   # (N, n_pars)
nom_vals = nominal_pars["value"].values.astype(float)
nom_errs = nominal_pars["error"].values.astype(float)
free = ~nominal_pars["frozen"].values.astype(bool)

delta = vals - nom_vals                           # shift of each fit w.r.t. the nominal fit
sys_err = np.sqrt(np.mean(delta ** 2, axis=0))    # systematic error (rms of the shifts)

summary = pd.DataFrame({
    "nominal": nom_vals,
    "stat_err": nom_errs,
    "sys_err": sys_err,
}, index=nominal_pars["name"].values)

summary[free]

,nominal,stat_err,sys_err
tilt,8.576046e-03,1.983690e-02,2.207781e-03
norm,9.836408e-01,2.051630e-02,2.732901e-02
index,2.519417e+00,4.945564e-01,6.495770e-02
amplitude,6.034628e-11,2.247352e-11,4.650446e-10
lon_0,3.467694e+02,1.324827e-03,2.152196e-01
lat_0,-2.930320e-01,1.346121e-01,5.361160e-01
sigma,7.627088e-01,2.079686e-01,5.695856e-01
